In [1]:
import polars as pl
import numpy as np
from scipy import stats
import altair as alt

In [2]:
df = pl.scan_parquet("data/data_bias.parquet")

In [3]:
def ci_lower(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.pdf((1 + confidence) / 2, n-1)

    return mean - ci

def ci_upper(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.pdf((1 + confidence) / 2, n-1)

    return mean + ci

In [4]:
df.with_columns(((pl.col("adv distance") - pl.col("adv distance").mean()) / pl.col("adv distance").std()).over(["dataset", "bias_type"]))

In [5]:
cols = ["train acc", "train f1", "test acc", "test f1", 
               "adv acc", "adv f1", "adv distance", "clusters"]

def agg_metrics(cols):
    res = []
    for col in cols:
        res.extend([
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).std().alias(f"{col}_std"),
            pl.col(col).map_batches(ci_lower, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_lb"),
            pl.col(col).map_batches(ci_upper, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_ub"),
        ])
    return res

In [6]:
df_stats = df.select(pl.exclude("seed")).with_columns(
    (
        (pl.col("adv distance") - pl.col("adv distance").mean()) / pl.col("adv distance").std()
        ).over(["dataset", "bias_type"])
    ).group_by(
    # pl.col("distribution"),
    # pl.col("model"),
    # pl.col("seed"),
    pl.col("bias"),
    pl.col("dataset"),
    pl.col("bias_type"),
    
).agg(
    agg_metrics(cols)
).sort([
    # pl.col("distribution"),
    # pl.col("seed"),
    # pl.col("depth"),
    pl.col("bias")
])

In [7]:
df_stats.collect()

bias,dataset,bias_type,train acc_mean,train acc_std,train acc_ci_lb,train acc_ci_ub,train f1_mean,train f1_std,train f1_ci_lb,train f1_ci_ub,test acc_mean,test acc_std,test acc_ci_lb,test acc_ci_ub,test f1_mean,test f1_std,test f1_ci_lb,test f1_ci_ub,adv acc_mean,adv acc_std,adv acc_ci_lb,adv acc_ci_ub,adv f1_mean,adv f1_std,adv f1_ci_lb,adv f1_ci_ub,adv distance_mean,adv distance_std,adv distance_ci_lb,adv distance_ci_ub,clusters_mean,clusters_std,clusters_ci_lb,clusters_ci_ub
f64,str,str,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32
0.1,"""wine_feature2""","""oversampling""",0.999979,0.000361,0.999974,0.999984,0.999981,0.000331,0.999976,0.999986,0.89305,0.066634,0.892098,0.894003,0.892285,0.067298,0.891323,0.893247,0.447484,0.153809,0.445285,0.449682,0.35012,0.166678,0.347737,0.352503,0.036207,1.021031,0.021611,0.050803,1.62,0.645748,1.610769,1.629231
0.1,"""wine_feature8""","""undersampling""",0.9902,0.010706,0.990047,0.990353,0.990137,0.010718,0.989984,0.990291,0.88707,0.073206,0.886023,0.888116,0.885784,0.07509,0.884711,0.886858,0.417702,0.168436,0.415294,0.420109,0.325458,0.173862,0.322973,0.327943,0.061018,0.982916,0.046967,0.075069,1.67,0.590703,1.661556,1.678444
0.1,"""wine_feature1""","""undersampling""",0.988453,0.010867,0.988297,0.988608,0.988339,0.011148,0.98818,0.988499,0.883257,0.074902,0.882186,0.884328,0.881782,0.076917,0.880682,0.882881,0.438573,0.167454,0.436179,0.440967,0.338045,0.177242,0.335511,0.340579,0.106941,1.058804,0.091805,0.122077,1.626667,0.578854,1.618392,1.634942
0.1,"""Car Evaluation_feature3""","""oversampling""",0.926304,0.018238,0.926078,0.926529,0.839241,0.037444,0.838777,0.839704,0.872988,0.035887,0.872544,0.873433,0.746405,0.073042,0.7455,0.74731,0.559323,0.1075,0.557992,0.560655,0.274,0.073962,0.273084,0.274916,0.043463,1.150899,0.029209,0.057717,9.265,2.658466,9.232074,9.297926
0.1,"""Car Evaluation_feature2""","""undersampling""",0.903553,0.05333,0.902892,0.904213,0.804357,0.057712,0.803642,0.805071,0.847632,0.061361,0.846872,0.848392,0.697562,0.093373,0.696406,0.698719,0.569877,0.103523,0.568595,0.571159,0.284423,0.088653,0.283325,0.285521,0.065551,1.153913,0.051259,0.079842,8.9925,3.29995,8.951629,9.033371
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.9,"""Car Evaluation_feature1""","""oversampling""",0.920302,0.020162,0.920053,0.920552,0.828271,0.041887,0.827753,0.82879,0.859919,0.037588,0.859454,0.860385,0.727397,0.076497,0.72645,0.728345,0.559327,0.095731,0.558142,0.560513,0.273889,0.068555,0.27304,0.274739,-0.091457,0.798709,-0.101349,-0.081564,9.5475,2.598003,9.515323,9.579678
0.9,"""wine_feature0""","""undersampling""",0.82382,0.069885,0.822821,0.824819,0.800085,0.085799,0.798859,0.801312,0.763845,0.112121,0.762242,0.765448,0.73391,0.137066,0.731951,0.73587,0.42427,0.167192,0.42188,0.42666,0.337014,0.174491,0.334519,0.339508,-0.068304,0.983231,-0.082359,-0.054248,1.63,0.583869,1.621653,1.638347
0.9,"""wine_feature2""","""undersampling""",0.82293,0.08225,0.821738,0.824122,0.797577,0.101106,0.796112,0.799042,0.758528,0.118722,0.756808,0.760248,0.727239,0.146118,0.725122,0.729357,0.470017,0.165559,0.467619,0.472416,0.380463,0.179737,0.377858,0.383067,-0.164164,0.933484,-0.17769,-0.150639,1.626712,0.615693,1.617791,1.635633


In [8]:
base = alt.Chart(
    df_stats.collect()
)

scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "clusters_mean"], 
    range=["blue", "red", "purple"]
)


dash_scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "clusters_mean"],
    range=[[2, 4], [1, 0], [8, 4]]  # [dash_length, gap_length]
)

# Chart for accuracy metrics (left y-axis)
acc_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["train acc_mean", "test acc_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y('value:Q').title("Accuracy").scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)
)


# Chart for adversarial distance (right y-axis)
adv_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["clusters_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y('value:Q').title("Adversarial Distance").axis(orient='right').scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)

)

ci_band0 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("train acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("train acc_ci_ub:Q"),
    color = alt.value("blue"),
)

ci_band1 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("test acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("test acc_ci_ub:Q"),
    color = alt.value("red")
)

ci_band2 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("adv distance_ci_lb:Q").axis(orient="right"),
    y2=alt.Y2("adv distance_ci_ub:Q"),
    color = alt.value("purple")
)

alt.layer(acc_chart + ci_band0 + ci_band1, adv_chart + ci_band2).resolve_scale(
    y='independent'
).facet(
    row="dataset",
    column="bias_type"
).resolve_scale(
    x="independent",
    y="independent"
)


alt.FacetChart(...)